# vepyr — Build Ensembl VEP Cache

Download an Ensembl VEP offline cache and convert it to optimized Parquet files and fjall KV stores.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
import os

os.environ["RUST_LOG"] = "info"

In [ ]:
from vepyr import build_cache

In [ ]:
!pip install ipywidgets jupyter tqdm

In [ ]:
# Configure
RELEASE = 115
CACHE_DIR = "/tmp/vepyr_cache"
SPECIES = "homo_sapiens"
ASSEMBLY = "GRCh38"
CACHE_TYPE = "vep"  # "vep", "merged", or "refseq"


# Set to an existing unpacked VEP cache directory to skip download, e.g.:
# LOCAL_CACHE = "/path/to/homo_sapiens/115_GRCh38"
LOCAL_CACHE = None

In [ ]:
results = build_cache(
    release=RELEASE,
    cache_dir=CACHE_DIR,
    species=SPECIES,
    assembly=ASSEMBLY,
    cache_type=CACHE_TYPE,
    local_cache=LOCAL_CACHE,
    partitions=6,
)

In [ ]:
import os

for path, rows in results:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    name = os.path.basename(path)
    print(f"{name:45s} {rows:>12,} rows  {size_mb:8.1f} MB")

In [ ]:
import pyarrow.parquet as pq

# Inspect one of the generated files
variation_file = [p for p, _ in results if "variation" in p][0]
table = pq.read_table(variation_file)
print(f"Schema: {table.schema}")
print(f"Rows: {table.num_rows:,}")
table.to_pandas().head()